# Official polyp dataset download and verification

Fresh archives are stored separately in Google Drive. Existing research files are never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import json, hashlib, urllib.request, zipfile, tarfile
from datetime import datetime, timezone
from PIL import Image
DRIVE_ROOT=Path('/content/drive/MyDrive')
AUDIT_ROOT=DRIVE_ROOT/'official-polyp-datasets-audit'
ARCHIVE_ROOT= AUDIT_ROOT/'archives'; EXTRACT_ROOT=AUDIT_ROOT/'extracted'; REPORT_ROOT=AUDIT_ROOT/'reports'
for x in (ARCHIVE_ROOT,EXTRACT_ROOT,REPORT_ROOT): x.mkdir(parents=True,exist_ok=True)
print('AUDIT_ROOT=',AUDIT_ROOT); print('DRIVE_MOUNT_GATE=PASS')

## Official source registry

Leave `archive_url` blank for manual/authenticated downloads. Direct URLs must come from official release pages.

In [ ]:
DATASETS={
'kvasir_seg':{'landing_page':'https://datasets.simula.no/kvasir-seg/','expected_images':1000,'expected_masks':1000,'archive_url':''},
'cvc_clinicdb':{'landing_page':'https://polyp.grand-challenge.org/CVCClinicDB/','expected_images':612,'expected_masks':612,'archive_url':''},
'cvc_colondb':{'landing_page':'http://mv.cvc.uab.es/projects/colon-qa/cvc-colondb/','expected_images':380,'expected_masks':380,'archive_url':''},
'cvc_300':{'landing_page':'https://polyp.grand-challenge.org/CVC300/','expected_images':60,'expected_masks':60,'archive_url':''},
'etis_laribpolypdb':{'landing_page':'https://polyp.grand-challenge.org/EtisLarib/','expected_images':196,'expected_masks':196,'archive_url':''}}

def sha256_file(path):
 h=hashlib.sha256(); n=0
 with path.open('rb') as f:
  while chunk:=f.read(8*1024*1024): h.update(chunk); n+=len(chunk)
 return h.hexdigest(),n
records=[]
for name,spec in DATASETS.items():
 d=ARCHIVE_ROOT/name; d.mkdir(parents=True,exist_ok=True)
 if spec['archive_url']:
  target=d/spec['archive_url'].rstrip('/').split('/')[-1]; urllib.request.urlretrieve(spec['archive_url'],target)
 for path in sorted(d.iterdir()):
  if path.is_file() and not path.name.endswith('.sha256'):
   h,n=sha256_file(path); records.append({'dataset':name,'path':str(path),'url':spec['archive_url'],'sha256':h,'bytes':n})
print(json.dumps(records,indent=2))

In [ ]:
def extract(record):
 a=Path(record['path']); out=EXTRACT_ROOT/record['dataset']; out.mkdir(parents=True,exist_ok=True)
 if a.name.lower().endswith('.zip'):
  with zipfile.ZipFile(a) as z: z.extractall(out)
 elif a.name.lower().endswith(('.tar','.tar.gz','.tgz','.tar.bz2')):
  with tarfile.open(a) as t: t.extractall(out)
 return out
for r in records: r['extract_root']=str(extract(r))
EXTS={'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.webp'}
reports=[]
for name,spec in DATASETS.items():
 imgs=[]; masks=[]; errors=[]
 for path in sorted((EXTRACT_ROOT/name).rglob('*')):
  if not path.is_file() or path.suffix.lower() not in EXTS: continue
  kind='mask' if any(x in str(path).lower() for x in ('mask','ground_truth','ground-truth','gt')) else 'image'
  try:
   h,n=sha256_file(path)
   with Image.open(path) as im:
    im.load(); item={'path':str(path),'sha256':h,'bytes':n,'dimensions':list(im.size),'mode':im.mode}
    if kind=='mask': item['values']=sorted(set(im.convert('L').getdata()))
   (masks if kind=='mask' else imgs).append(item)
  except Exception as e: errors.append({'path':str(path),'error':str(e)})
 reports.append({'dataset':name,'expected_images':spec['expected_images'],'expected_masks':spec['expected_masks'],'image_count':len(imgs),'mask_count':len(masks),'images':imgs,'masks':masks,'errors':errors})
 print(name,len(imgs),'/',spec['expected_images'],len(masks),'/',spec['expected_masks'])

In [ ]:
all_items=[x for r in reports for x in r['images']+r['masks']]; index={}
for x in all_items: index.setdefault((x['sha256'],x['bytes']),[]).append(x['path'])
status={'archive_records':records,'datasets':reports,'duplicate_groups':[v for v in index.values() if len(v)>1],'official_byte_authenticity':False,'provenance_status':'PARTIALLY_VERIFIED','integrity_status':'PASS' if all(r['image_count']==r['expected_images'] and r['mask_count']==r['expected_masks'] and not r['errors'] for r in reports) else 'REVIEW_REQUIRED','training_allowed':False,'limitations':['Official landing pages are not archive checksums.','Manual/authenticated archives must be documented.','Pairing and provenance need dataset-specific review.']}
ts=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); jp=REPORT_ROOT/f'official_dataset_audit_{ts}.json'; tp=REPORT_ROOT/f'official_dataset_audit_{ts}.txt'; jp.write_text(json.dumps(status,indent=2),encoding='utf-8'); tp.write_text(chr(10).join(['OFFICIAL POLYP DATASET AUDIT',f'Timestamp UTC: {ts}',f'Archive records: {len(records)}',f'Integrity status: {status["integrity_status"]}','Provenance status: PARTIALLY_VERIFIED','Training allowed: False'])+chr(10),encoding='utf-8'); print('JSON=',jp); print('TXT=',tp); print('TRAINING_ALLOWED=False')